In [1]:
import os

#data handling
import numpy as np
import pandas as pd
pd.set_option("display.max_rows", None, "display.max_columns", None)

#plotting
import plotly.express as px
import plotly.graph_objects as go
import nglview as nv
from ipywidgets import HBox, VBox
import glob

# atomic structures manipulation
import biobox as bb

#carbamylation analysis procedures
from uniprot import Uniprot
from protein import PDB
from measure import Measure
import postprocessing as postproc

In [2]:
outdir="test"

In [3]:
UP = Uniprot()
UP.get_protein_data("P22607") # example: load structures for a single UNIPROT
#UP.from_csv_file("inputs/kiba_short.csv") # add structures from a .csv file
UP.df

,Uniprot Entry,PDB Code,Method,Resolution,Chains
0,P22607,AF-P22607-F1-model_v1,Predicted,NaN,NaN
1,P22607,1RY7,X-ray,3.20,B
2,P22607,2LZL,NMR,-,A/B
3,P22607,4K33,X-ray,2.34,A
4,P22607,6LVM,X-ray,2.53,A
5,P22607,6PNX,X-ray,2.20,A/B
6,P22607,7DHL,X-ray,2.57,A


In [4]:
pdb = PDB(outdir=outdir, gap=10)

In [5]:
pdb.describe_dataset(UP.df)


UNIPROT: P22607, PDB: AF-P22607-F1-model_v1
> downloading AlphaFold structure AF-P22607-F1-model_v1
Download success
test/curated/AF-P22607-F1-model_v1.pdb


FileNotFoundError: [Errno 2] No such file or directory: 'test/curated/AF-P22607-F1-model_v1.pdb'

In [6]:
pwd

'/Users/toni_brain/Projects_git/carbamylation'

In [50]:
def gather_AF_data(PDBCODE, skip_if_found=True):
    

    # getting confidence
    f = open(PDBCODE)
    data = f.readlines()
    f.close()
    confidence_score = []
    info_dic = {}
    for l in data:
        if l.startswith('ATOM'):
            curr_line =l.split()
            confidence_score.append(float(curr_line[-2]))
    info_dic['mean'] = np.mean(confidence_score)
    info_dic['std'] = np.std(confidence_score)
    info_dic['name'] = PDBCODE
    
    # getting FASTA sequence
    seq_list = []
    env = Environ()
    mdl = Model(env, file='result/curated/AF-P22607-F1-model_v1.pdb')
    aln = Alignment(env)
    aln.append_model(mdl, align_codes='result/curated/AF-P22607-F1-model_v1.pdb')
    for seq in aln:
        for r in seq.residues:
            seq_list.append(r.name)
    info_dic['seq_length'] = len(seq_list)
    info_dic['seq'] = seq_list
    return info_dic

In [51]:
info = gather_AF_data('result/curated/AF-P22607-F1-model_v1.pdb')

In [52]:
info


{'mean': 75.36963816323218,
 'std': 22.66797173349757,
 'name': 'result/curated/AF-P22607-F1-model_v1.pdb',
 'seq_length': 806,
 'seq': ['MET',
  'GLY',
  'ALA',
  'PRO',
  'ALA',
  'CYS',
  'ALA',
  'LEU',
  'ALA',
  'LEU',
  'CYS',
  'VAL',
  'ALA',
  'VAL',
  'ALA',
  'ILE',
  'VAL',
  'ALA',
  'GLY',
  'ALA',
  'SER',
  'SER',
  'GLU',
  'SER',
  'LEU',
  'GLY',
  'THR',
  'GLU',
  'GLN',
  'ARG',
  'VAL',
  'VAL',
  'GLY',
  'ARG',
  'ALA',
  'ALA',
  'GLU',
  'VAL',
  'PRO',
  'GLY',
  'PRO',
  'GLU',
  'PRO',
  'GLY',
  'GLN',
  'GLN',
  'GLU',
  'GLN',
  'LEU',
  'VAL',
  'PHE',
  'GLY',
  'SER',
  'GLY',
  'ASP',
  'ALA',
  'VAL',
  'GLU',
  'LEU',
  'SER',
  'CYS',
  'PRO',
  'PRO',
  'PRO',
  'GLY',
  'GLY',
  'GLY',
  'PRO',
  'MET',
  'GLY',
  'PRO',
  'THR',
  'VAL',
  'TRP',
  'VAL',
  'LYS',
  'ASP',
  'GLY',
  'THR',
  'GLY',
  'LEU',
  'VAL',
  'PRO',
  'SER',
  'GLU',
  'ARG',
  'VAL',
  'LEU',
  'VAL',
  'GLY',
  'PRO',
  'GLN',
  'ARG',
  'LEU',
  'GLN',
  'VAL',
 

In [46]:
info

{'mean': 75.36963816323218,
 'std': 22.66797173349757,
 'name': 'result/curated/AF-P22607-F1-model_v1.pdb'}

In [7]:
import numpy as np
import biobox as bb
from copy import deepcopy

from modeller import *
from modeller.automodel import *

In [12]:
env = Environ()
mdl = Model(env, file='result/curated/AF-P22607-F1-model_v1.pdb')
aln = Alignment(env)
aln.append_model(mdl, align_codes='result/curated/AF-P22607-F1-model_v1.pdb')

1

In [30]:
for seq in aln:
    for r in seq.residues:
        print(r.name)
    print("   Sequence %s from %s contains %d residues" \
          % (seq.code, seq.residues, len(seq)))

MET
GLY
ALA
PRO
ALA
CYS
ALA
LEU
ALA
LEU
CYS
VAL
ALA
VAL
ALA
ILE
VAL
ALA
GLY
ALA
SER
SER
GLU
SER
LEU
GLY
THR
GLU
GLN
ARG
VAL
VAL
GLY
ARG
ALA
ALA
GLU
VAL
PRO
GLY
PRO
GLU
PRO
GLY
GLN
GLN
GLU
GLN
LEU
VAL
PHE
GLY
SER
GLY
ASP
ALA
VAL
GLU
LEU
SER
CYS
PRO
PRO
PRO
GLY
GLY
GLY
PRO
MET
GLY
PRO
THR
VAL
TRP
VAL
LYS
ASP
GLY
THR
GLY
LEU
VAL
PRO
SER
GLU
ARG
VAL
LEU
VAL
GLY
PRO
GLN
ARG
LEU
GLN
VAL
LEU
ASN
ALA
SER
HIS
GLU
ASP
SER
GLY
ALA
TYR
SER
CYS
ARG
GLN
ARG
LEU
THR
GLN
ARG
VAL
LEU
CYS
HIS
PHE
SER
VAL
ARG
VAL
THR
ASP
ALA
PRO
SER
SER
GLY
ASP
ASP
GLU
ASP
GLY
GLU
ASP
GLU
ALA
GLU
ASP
THR
GLY
VAL
ASP
THR
GLY
ALA
PRO
TYR
TRP
THR
ARG
PRO
GLU
ARG
MET
ASP
LYS
LYS
LEU
LEU
ALA
VAL
PRO
ALA
ALA
ASN
THR
VAL
ARG
PHE
ARG
CYS
PRO
ALA
ALA
GLY
ASN
PRO
THR
PRO
SER
ILE
SER
TRP
LEU
LYS
ASN
GLY
ARG
GLU
PHE
ARG
GLY
GLU
HIS
ARG
ILE
GLY
GLY
ILE
LYS
LEU
ARG
HIS
GLN
GLN
TRP
SER
LEU
VAL
MET
GLU
SER
VAL
VAL
PRO
SER
ASP
ARG
GLY
ASN
TYR
THR
CYS
VAL
VAL
GLU
ASN
LYS
PHE
GLY
SER
ILE
ARG
GLN
THR
TYR
THR
LEU
ASP
VAL
LEU
GLU
ARG
SER
PRO


In [4]:
for index, row in UP.df.iterrows():
    PDBCODE = row["PDB Code"]
    uniprot_code = row["Uniprot Entry"]

    print("\nUNIPROT: %s, PDB: %s" % (uniprot_code, PDBCODE))
    
    if PDBCODE[:2] == "AF": 
        # gather AF data
        fasta, len_fasta, mean_confidence, std_confidence = gather_AF_data(PDBCODE)


UNIPROT: P22607, PDB: AF-P22607-F1-model_v1

UNIPROT: P22607, PDB: 1RY7

UNIPROT: P22607, PDB: 2LZL

UNIPROT: P22607, PDB: 4K33

UNIPROT: P22607, PDB: 6LVM

UNIPROT: P22607, PDB: 6PNX

UNIPROT: P22607, PDB: 7DHL


In [4]:
pdb = PDB(outdir=outdir, gap=10)

In [5]:
pdb.clean_and_split_pdb('7DHL')

> downloading PDB 7DHL
> downloading FASTA for 7DHL
There should be 755 in this sequence
(4,)
[4, 469, 459]
>> Patching failed for conformer 0. large gap detected (459 residues)


/Users/toni_brain/miniconda3/envs/carbamy/lib/python3.10/site-packages/numpy/core/fromnumeric.py:2007: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  result = asarray(a).shape


Exception: Patching failed for all conformers